In [6]:
import pandas as pd
import re
import os

def limpar_avaliacoes(texto):
    if isinstance(texto, str) == False:
        return texto

    # --- 1. Padronização de Espaços e Pontuações ---
    
    # Remove espaços duplos ou múltiplos no meio do texto e nas pontas
    texto = re.sub(r'\s{2,}', ' ', texto).strip()

    # Tira o espaço antes de pontuações (ex: "prazo ." -> "prazo.")
    texto = re.sub(r'\s+([.,;:!?])', r'\1', texto)

    # Reduz repetições de [.,;:] (ex: "ruim..." -> "ruim.")
    texto = re.sub(r'([.,;:])\1+', r'\1', texto)

    # Garante um único espaço após pontuação (ex: "oi,tudo" -> "oi, tudo")
    texto = re.sub(r'([.,;:])(?=[^\s])', r'\1 ', texto)

    # --- 2. Correção de Siglas e Abreviações ---

    mapeamento = {
        r'\b(ñ|nao|nã)\b': 'não',
        r'\bvc\b': 'você',
        r'\bvcs\b': 'vocês',
        r'\b(mt|mto)\b': 'muito'
    }

    def aplicar_mapeamento(match):
        original = match.group(0)
        alvo = ""
        
        for padrao, subst in mapeamento.items():
            if re.search(padrao, original, flags=re.IGNORECASE):
                alvo = subst
                break
        
        if original.isupper():
            return alvo.upper()
        if original[0].isupper():
            return alvo.capitalize()
        return alvo

    regex_completo = '|'.join(mapeamento.keys())
    texto = re.sub(regex_completo, aplicar_mapeamento, texto, flags=re.IGNORECASE)

    # --- 3. Ajustes Finais de Texto ---

    # Letra maiúscula após o ponto final
    texto = re.sub(r'(\.\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), texto)

    return texto

def processar_csv(caminho_arquivo, nome_coluna):
    if os.path.exists(caminho_arquivo) == False:
        print(f"Erro: Arquivo não encontrado em {caminho_arquivo}")
        return

    df = pd.read_csv(caminho_arquivo)

    if (nome_coluna in df.columns):
        print(f"Limpando espaços e padronizando a coluna '{nome_coluna}'...")
        
        df[nome_coluna] = df[nome_coluna].apply(limpar_avaliacoes)
        
        df.to_csv(caminho_arquivo, index=False)
        print(f"Sucesso! Espaços duplos removidos e arquivo atualizado.")
    else:
        print(f"Erro: A coluna '{nome_coluna}' não existe no arquivo CSV.")

# Execução
processar_csv('./Banco_de_Dados_PII3_AWS/avaliacoes.csv', 'review_comment_title')

Limpando espaços e padronizando a coluna 'review_comment_title'...
Sucesso! Espaços duplos removidos e arquivo atualizado.


In [10]:
def converter_para_string(df, nome_coluna):
    """
    Converte uma coluna do tipo object para string.
    """
    if nome_coluna in df.columns:
        df[nome_coluna] = df[nome_coluna].astype("string")
        print(f"Coluna '{nome_coluna}' convertida para string.")
    else:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
    return df

df = pd.read_csv('./Banco_de_Dados_PII3_AWS/avaliacoes.csv')

converter_para_string(df, 'order_id')

Coluna 'order_id' convertida para string.


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [12]:
def converter_para_datetime(df, nome_coluna):
    """
    Converte uma coluna do tipo object para datetime.
    """
    if nome_coluna in df.columns:
        # errors='coerce' transforma valores inválidos em nulos (NaT)
        df[nome_coluna] = pd.to_datetime(df[nome_coluna], errors='coerce')
        print(f"Coluna '{nome_coluna}' convertida para datetime.")
    else:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
    return df

converter_para_datetime(df, 'review_answer_timestamp')

Coluna 'review_answer_timestamp' convertida para datetime.


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01,2018-07-02 12:59:13
